In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal, stats

# --- CONFIGURATION ---
DATA_PATH = "S2.pkl"
OUT = "outputs_analysis"
os.makedirs(OUT, exist_ok=True)

WIN = 30  # 30-second non-overlapping windows
FEATURES = ["HR", "EDA", "BVP_amp", "TEMP"]
LABELS_MAP = {1: "Baseline", 2: "Stress", 3: "Amusement", 4: "Meditation"}

print("Loading WESAD S2 dataset...")
with open(DATA_PATH, "rb") as f:
    data = pickle.load(f, encoding='latin1')

wrist = data['signal']['wrist']
eda_raw = wrist['EDA'].flatten()
temp_raw = wrist['TEMP'].flatten()
bvp_raw = wrist['BVP'].flatten()
labels_700hz = data['label'].flatten()

fs_eda = 4.0
fs_temp = 4.0
fs_bvp = 64.0
fs_label = 700.0

t_eda = np.arange(len(eda_raw)) / fs_eda
t_temp = np.arange(len(temp_raw)) / fs_temp
t_bvp = np.arange(len(bvp_raw)) / fs_bvp

# Map 700Hz labels to 4Hz EDA timeline
label_indices = np.clip((t_eda * fs_label).astype(int), 0, len(labels_700hz) - 1)
labels_aligned = labels_700hz[label_indices]

sig = {
    "EDA": {"fs": fs_eda, "raw": eda_raw, "t": t_eda},
    "TEMP": {"fs": fs_temp, "raw": temp_raw, "t": t_temp},
    "BVP": {"fs": fs_bvp, "raw": bvp_raw, "t": t_bvp}
}

# Estimate Heart Rate (HR) from BVP peaks
peaks, _ = signal.find_peaks(bvp_raw, distance=fs_bvp*0.4, prominence=15)
peak_times = peaks / fs_bvp
hr_interp = np.interp(t_eda, peak_times[1:], 60.0 / np.diff(peak_times))
sig["HR"] = {"fs": fs_eda, "raw": hr_interp, "t": t_eda}

# --- 1. SIGNAL CLEANING ---
log = {}
# TEMP cleaning
tmp = sig["TEMP"]["raw"].copy()
bad_temp = (tmp < 25) | (tmp > 42)
log["TEMP_outliers"] = int(bad_temp.sum())
tmp[bad_temp] = np.nan
tmp = pd.Series(tmp).interpolate(limit_direction="both").to_numpy()
sig["TEMP"]["clean"] = signal.medfilt(tmp, 5)

# EDA cleaning
eda = sig["EDA"]["raw"].copy()
bad_eda = eda < 0.01
log["EDA_contact_loss"] = int(bad_eda.sum())
eda[bad_eda] = np.nan
eda = pd.Series(eda).interpolate(limit_direction="both").to_numpy()
b, a = signal.butter(2, 1.0 / (fs_eda / 2), "low")
sig["EDA"]["clean"] = signal.filtfilt(b, a, eda)

# HR cleaning
hr = sig["HR"]["raw"].copy()
bad_hr = (hr < 40) | (hr > 200)
log["HR_implausible"] = int(bad_hr.sum())
hr[bad_hr] = np.nan
hr = pd.Series(hr).interpolate(limit_direction="both").to_numpy()
sig["HR"]["clean"] = signal.medfilt(hr, 5)

# BVP filtering (bandpass 0.5 - 8 Hz)
b, a = signal.butter(3, [0.5 / (fs_bvp/2), 8 / (fs_bvp/2)], "band")
sig["BVP"]["clean"] = signal.filtfilt(b, a, bvp_raw)

# --- 2. DETRENDING ---
for k in ["EDA", "TEMP"]:
    x = sig[k]["clean"]
    sig[k]["detr"] = signal.detrend(x, type="linear") + x.mean()
sig["HR"]["detr"] = sig["HR"]["clean"]
sig["BVP"]["detr"] = sig["BVP"]["clean"]

# --- 3. WINDOWING & FEATURE EXTRACTION ---
rows = []
n_windows = int(t_eda[-1] // WIN)
for w in range(n_windows):
    a0, a1 = w * WIN, (w + 1) * WIN
    mask_lab = (t_eda >= a0) & (t_eda < a1)
    if not mask_lab.any(): continue
    win_labels = labels_aligned[mask_lab]
    dom_label = int(stats.mode(win_labels, keepdims=True).mode[0])
    if dom_label not in LABELS_MAP: continue
    
    cat_name = LABELS_MAP[dom_label]
    r = dict(window_id=w, category=cat_name, t_start=a0)
    for k in ["HR", "EDA", "TEMP"]:
        m = (sig[k]["t"] >= a0) & (sig[k]["t"] < a1)
        r[k] = sig[k]["detr"][m].mean()
    m_bvp = (sig["BVP"]["t"] >= a0) & (sig["BVP"]["t"] < a1)
    r["BVP_amp"] = np.sqrt(np.mean(sig["BVP"]["detr"][m_bvp] ** 2)) if m_bvp.any() else 0
    rows.append(r)

df = pd.DataFrame(rows)
df["stress_binary"] = np.where(df.category == "Stress", "Stress", "Non-stress")
df.to_csv(f"{OUT}/window_features.csv", index=False)

# --- 4. DESCRIPTIVE STATISTICS ---
cats = ["Baseline", "Stress", "Amusement", "Meditation"]
desc_cat = df.groupby("category")[FEATURES].agg(["count", "mean", "median", "std", "min", "max"]).reindex(cats)
desc_cat.round(3).to_csv(f"{OUT}/descriptives_by_category.csv")

desc_bin = df.groupby("stress_binary")[FEATURES].agg(["count", "mean", "median", "std", "min", "max"])
desc_bin.round(3).to_csv(f"{OUT}/descriptives_stress_vs_nonstress.csv")

# --- 5. HYPOTHESIS TESTING ---
def cohen_d(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    return (x.mean() - y.mean()) / sp if sp > 0 else 0

res_t, res_a = [], []
for f in FEATURES:
    st = df.loc[df.stress_binary == "Stress", f].to_numpy()
    ns = df.loc[df.stress_binary == "Non-stress", f].to_numpy()
    if len(st) > 1 and len(ns) > 1:
        t, p = stats.ttest_ind(st, ns, equal_var=False)
        res_t.append(dict(feature=f, mean_stress=st.mean(), mean_non=ns.mean(), t_stat=t, p_value=p, cohen_d=cohen_d(st, ns)))
    
    groups = [df.loc[df.category == c, f].to_numpy() for c in cats if len(df.loc[df.category == c, f]) > 1]
    if len(groups) > 1:
        F, pa = stats.f_oneway(*groups)
        res_a.append(dict(feature=f, F_stat=F, p_value=pa))

pd.DataFrame(res_t).round(4).to_csv(f"{OUT}/ttest_stress_vs_nonstress.csv", index=False)
pd.DataFrame(res_a).round(4).to_csv(f"{OUT}/anova_four_categories.csv", index=False)
print("Analysis completed successfully. Check output folder:", OUT)

Loading WESAD S2 dataset...
Analysis completed successfully. Check output folder: outputs_analysis


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the windowed features generated by your script
df = pd.read_csv("outputs_analysis/window_features.csv")
print(df.head())

   window_id  category  t_start         HR       EDA       TEMP     BVP_amp  \
0         10  Baseline      300  90.799069  1.233192  34.867984  110.642252   
1         11  Baseline      330  87.172308  1.015709  34.862445  123.569389   
2         12  Baseline      360  81.218569  0.822950  34.786906   46.991757   
3         13  Baseline      390  78.633960  0.572758  34.785367   37.330679   
4         14  Baseline      420  74.808034  0.452273  34.843161   39.823831   

  stress_binary  
0    Non-stress  
1    Non-stress  
2    Non-stress  
3    Non-stress  
4    Non-stress  


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ["HR", "EDA", "BVP_amp", "TEMP"]
titles = ["Heart Rate (HR)", "Electrodermal Activity (EDA)", "BVP Amplitude", "Skin Temperature"]
palette = {"Baseline": "#4C72B0", "Stress": "#C44E52", "Amusement": "#DD8452", "Meditation": "#55A868"}

for ax, feature, title in zip(axes.flatten(), features, titles):
    sns.boxplot(x="category", y=feature, data=df, palette=palette, ax=ax, order=["Baseline", "Stress", "Amusement", "Meditation"])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel("Condition / Category", fontsize=10)
    ax.set_ylabel(feature, fontsize=10)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle("Physiological Features Across Stress & Affective Categories", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("outputs_analysis/anova_boxplots.png", dpi=300, bbox_inches='tight')
plt.show()

C:\Users\Ishita\AppData\Local\Temp\ipykernel_17700\77877208.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="category", y=feature, data=df, palette=palette, ax=ax, order=["Baseline", "Stress", "Amusement", "Meditation"])
C:\Users\Ishita\AppData\Local\Temp\ipykernel_17700\77877208.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="category", y=feature, data=df, palette=palette, ax=ax, order=["Baseline", "Stress", "Amusement", "Meditation"])
C:\Users\Ishita\AppData\Local\Temp\ipykernel_17700\77877208.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effe

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ["HR", "EDA", "BVP_amp", "TEMP"]
titles = ["Heart Rate (HR)", "Electrodermal Activity (EDA)", "BVP Amplitude", "Skin Temperature"]
palette = {"Baseline": "#4C72B0", "Stress": "#C44E52", "Amusement": "#DD8452", "Meditation": "#55A868"}

for ax, feature, title in zip(axes.flatten(), features, titles):
    sns.boxplot(
        x="category", y=feature, data=df, 
        hue="category", palette=palette, legend=False, 
        ax=ax, order=["Baseline", "Stress", "Amusement", "Meditation"]
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel("Condition / Category", fontsize=10)
    ax.set_ylabel(feature, fontsize=10)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle("Physiological Features Across Stress & Affective Categories", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("outputs_analysis/anova_boxplots.png", dpi=300, bbox_inches='tight')
plt.show()

C:\Users\Ishita\AppData\Local\Temp\ipykernel_17700\3275003160.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, feature in zip(axes, features):
    sns.barplot(
        x="stress_binary", y=feature, data=df, 
        hue="stress_binary", palette=["#4C72B0", "#C44E52"], 
        errorbar=('ci', 68), legend=False, 
        ax=ax, order=["Non-stress", "Stress"]
    )
    ax.set_title(f"{feature} (Stress vs Non-Stress)", fontsize=10, fontweight='bold')
    ax.set_xlabel("")
    ax.set_ylabel(feature)

plt.suptitle("Mean Comparison: Stress vs. Non-Stress Conditions", fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig("outputs_analysis/ttest_barplots.png", dpi=300, bbox_inches='tight')
plt.show()

C:\Users\Ishita\AppData\Local\Temp\ipykernel_17700\3140385707.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print("--- T-TEST RESULTS (Stress vs Non-Stress) ---")
ttest_res = pd.read_csv("outputs_analysis/ttest_stress_vs_nonstress.csv")
display(ttest_res)

print("\n--- ONE-WAY ANOVA RESULTS (Across 4 Categories) ---")
anova_res = pd.read_csv("outputs_analysis/anova_four_categories.csv")
display(anova_res)

--- T-TEST RESULTS (Stress vs Non-Stress) ---


,feature,mean_stress,mean_non,t_stat,p_value,cohen_d
0,HR,74.9285,72.4946,1.2345,0.2289,0.3812
1,EDA,0.8334,0.2482,10.4673,0.0000,2.4739
2,BVP_amp,60.3881,38.2599,2.7350,0.0105,0.7001
3,TEMP,33.2878,34.9162,-17.3729,0.0000,-5.9465



--- ONE-WAY ANOVA RESULTS (Across 4 Categories) ---


,feature,F_stat,p_value
0,HR,10.0407,0.0000
1,EDA,41.8620,0.0000
2,BVP_amp,6.5527,0.0005
3,TEMP,220.8910,0.0000
